# MLServer

A comprehensive guide to MLServer for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

**MLServer** is a lightweight, extensible inference server for machine learning models. It was originally developed by Seldon and is now widely used as a **generic model runtime** in platforms like KServe.

Instead of writing and maintaining your own HTTP server for every model, you package models in a standard way and let MLServer handle **lifecycle, request/response protocols, and observability**.

### What is it?

At a high level, MLServer is:

- A **Python-based model server** process that can host one or more models.
- A set of **runtimes** for popular frameworks (scikit‑learn, XGBoost, LightGBM, PyTorch, TensorFlow, ONNX, etc.).
- A **configuration-driven** system: you describe models using `model-settings.json` and `settings.json` files.
- An implementation of the **V2 Inference Protocol** (and others), making it compatible with modern inference platforms.

You can run MLServer as a standalone container or as the serving runtime underneath a higher-level system like KServe.

### Why use it?

Key benefits of using MLServer:

- **Multi-framework support**  
  Use built-in runtimes for many common libraries, or write a custom runtime when needed.
- **Standardized inference protocols**  
  Supports the V2 Inference API used by several modern serving stacks, simplifying integrations.
- **Configuration over boilerplate**  
  Declare models and settings in JSON/YAML instead of hand-writing REST servers.
- **Composable with orchestration platforms**  
  Works well inside Kubernetes via KServe or custom deployments.
- **Python-friendly extensibility**  
  Custom logic (pre/post-processing, business rules) can be written in Python.

### When to use it?

MLServer is particularly useful when:

- You need a **general-purpose model server** that supports multiple frameworks.
- You are deploying models on **KServe** and want to understand or customize the underlying runtime.
- You have many relatively small models and want to reduce per-model boilerplate.
- You prefer a **configuration-driven** approach but still want the option to extend behavior in Python.

You might choose alternatives when:

- You are focused on **LLM-specific serving** (vLLM, TGI, BentoML/OpenLLM, etc.).
- You need extreme low-level control over GPU kernels and batching (Triton Inference Server).
- You want a full **application framework** with routing, auth, and complex business logic built-in (FastAPI, BentoML, Mosec).

## Key Features

### Core capabilities of MLServer

| Feature                         | Description                                                           | Benefit                                           |
|---------------------------------|-----------------------------------------------------------------------|---------------------------------------------------|
| **Multi-framework runtimes**    | Built-in support for SKLearn, XGBoost, LightGBM, ONNX, PyTorch, etc. | Serve diverse models with a single infrastructure |
| **V2 Inference Protocol**       | Implements the standardized inference API used by modern platforms.  | Easier interoperability with gateways/orchestrators |
| **Configuration-driven models** | Models described via `model-settings.json` and `settings.json`.      | Less boilerplate code, consistent deployments     |
| **Multi-model serving**         | Host multiple models in one MLServer process.                        | Reduced operational overhead                      |
| **Dynamic model loading**       | Load/unload models without restarting the entire server.             | Faster iteration and better resource utilization  |
| **Python extensibility**        | Custom runtimes and hooks implemented in Python.                     | Flexible pre/post-processing and business logic   |
| **Metrics and logging hooks**   | Integrations for Prometheus-style metrics and structured logging.    | Production-grade observability                    |

### MLOps-friendly properties

- **Container-first**: Easy to run as a Docker image or inside Kubernetes.
- **Composable**: Often used under the hood by higher-level systems like KServe.
- **Stateless**: Model artifacts live on mounted volumes or object stores; servers are easy to scale horizontally.
- **Protocol-focused**: Because it speaks a standard inference protocol, it can plug into many different control planes.

## Architecture Overview

MLServer provides a thin, configurable serving layer between your **model artifacts** and your **client applications**.

```text
               ┌───────────────────────────────┐
               │       Client Apps             │
               │  • Web / mobile frontends     │
               │  • Backend services / APIs    │
               │  • Batch jobs / pipelines     │
               └───────────────┬───────────────┘
                               │  HTTP / gRPC (V2 Inference API)
                               ▼
                    ┌───────────────────────┐
                    │       MLServer        │
                    │   (model runtimes)    │
                    └─────────┬─────────────┘
                              │
                              ▼
                 ┌──────────────────────────┐
                 │   Model Repository       │
                 │  /models/                │
                 │    ├── model-a/          │
                 │    │    ├── model.pt     │
                 │    │    └── model-settings.json
                 │    └── model-b/          │
                 │         ├── model.onnx   │
                 │         └── model-settings.json
                 └──────────────────────────┘
```

### Components

1. **Model repository**
   - One directory per model, containing:
     - Model artifact (e.g., `model.pkl`, `model.onnx`, `model.pt`).
     - `model-settings.json` with metadata: name, implementation, parameters.

2. **MLServer process**
   - Python process that:
     - Reads `settings.json` (global) and `model-settings.json` (per model).
     - Instantiates the appropriate **runtime** (SKLearn, XGBoost, PyTorch, custom, …).
     - Exposes HTTP/gRPC endpoints implementing the V2 Inference API.

3. **Runtimes**
   - Pluggable classes implementing:
     - `load()` – load model artifact.
     - `predict()` / `infer()` – run inference for a request.
     - Optional hooks for pre/post-processing and batching.

4. **Clients**
   - Any system that can speak HTTP or gRPC:
     - Application backends
     - Airflow / Kubeflow / Argo steps
     - Batch pipelines (Spark, Beam, Python scripts)

5. **Optional control plane / orchestrator**
   - Systems like **KServe** or custom controllers:
     - Manage MLServer pods as part of a larger platform.
     - Configure routing, autoscaling, and security.

MLServer intentionally focuses on **model execution and protocol handling**, leaving higher-level concerns (auth, routing, A/B testing) to gateways and orchestrators.

## Installation

You can run MLServer either **directly with Python** (via `pip`) or **inside a container**. In production, the container approach is more common; in notebooks, `pip` is convenient.

### 1. Python / pip installation (local dev)

Requirements:

- Python 3.8+

Install the core server and a few runtimes:

```bash
pip install mlserver

# Add runtimes for specific frameworks (examples)
pip install mlserver-sklearn mlserver-xgboost
# or
pip install mlserver-mlflow mlserver-onnxruntime
```

This gives you the `mlserver` CLI plus the runtime packages.

### 2. Docker image (recommended for production)

MLServer publishes official images you can use in CI/CD and Kubernetes:

```bash
# Example (adjust tag as needed)
docker pull seldonio/mlserver:latest
```

You will typically:

- Build an image **containing your models and configuration**.
- Or mount a model directory into the generic MLServer image at runtime.

### 3. Configuration files

MLServer is configured via JSON/YAML files:

- `settings.json` – global server settings (HTTP/gRPC ports, logging, etc.).
- `model-settings.json` – per-model configuration (name, implementation, parameters).

In the next sections we’ll create a minimal `model-settings.json` and start MLServer against a local model directory.

In [ ]:
# Optional: install MLServer and common runtimes in this environment

# Uncomment if you are running this notebook in a fresh environment (e.g., Colab)
# and want to experiment with MLServer locally.

# !pip install mlserver
# !pip install mlserver-sklearn mlserver-xgboost
# # Add more runtimes as needed, for example:
# # !pip install mlserver-onnxruntime mlserver-mlflow

## Basic Usage

In this section we’ll walk through a minimal MLServer setup:

1. Train (or mock) a simple **scikit‑learn** model.
2. Create a `model-settings.json` file that tells MLServer how to load it.
3. Start MLServer pointing at the model directory.
4. Send a prediction request using the V2 Inference HTTP API.

This pattern generalizes to other runtimes (XGBoost, ONNX, PyTorch) with only small changes to the config.

In [ ]:
# Basic end-to-end example with MLServer (scikit-learn)

import json
import pathlib

import numpy as np

try:
    from sklearn.datasets import load_iris
    from sklearn.linear_model import LogisticRegression
    import joblib
except ImportError as e:
    print("scikit-learn and joblib are required for this example. Install with:")
    print("  pip install scikit-learn joblib")
    raise

# Paths
MODEL_DIR = pathlib.Path("models") / "iris-sklearn"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "model.joblib"
MODEL_SETTINGS_PATH = MODEL_DIR / "model-settings.json"

# 1. Train a tiny scikit-learn model
iris = load_iris()
X, y = iris.data, iris.target

clf = LogisticRegression(max_iter=200)
clf.fit(X, y)

joblib.dump(clf, MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")

# 2. Create model-settings.json for MLServer's SKLearn runtime
model_settings = {
    "name": "iris-sklearn",
    "implementation": "mlserver_sklearn.SKLearnModel",
    "parameters": {
        "uri": str(MODEL_PATH.name)
    }
}

MODEL_SETTINGS_PATH.write_text(json.dumps(model_settings, indent=2))
print(f"model-settings.json written to: {MODEL_SETTINGS_PATH}")

print("\nNext steps (run in a terminal, not inside this notebook):\n")
print("1) Install MLServer and sklearn runtime (if not already):")
print("   pip install mlserver mlserver-sklearn")
print("2) Start MLServer pointing at the models directory:")
print("   mlserver start models")
print("3) Send a V2 Inference request to:")
print("   POST http://localhost:8080/v2/models/iris-sklearn/infer")

## Advanced Features

MLServer includes several capabilities that become important in real-world MLOps setups.

### 1. Multi-model and dynamic loading

- A single MLServer process can host **multiple models**.
- You can:
  - Start the server with a directory containing multiple model folders.
  - Dynamically **load/unload models** via configuration or control-plane integrations.
- This is useful when you have many small/medium models (e.g., per-tenant models) and want to minimize infrastructure overhead.

### 2. Multiple runtimes in one server

- You can mix runtimes in a single deployment:
  - `mlserver-sklearn` for classical ML models.
  - `mlserver-xgboost` for gradient boosting.
  - `mlserver-onnxruntime` for ONNX-exported models.
  - `mlserver-mlflow` for MLflow-packaged models.
- Each model has its own `implementation` in `model-settings.json`.

### 3. Custom runtimes and hooks

- When built-in runtimes are not enough, you can:
  - Implement a **custom runtime** class in Python.
  - Add **pre-processing** and **post-processing** logic around the model.
- This lets you encapsulate feature engineering, validation, or business rules inside the serving layer.

### 4. Protocol-level flexibility

- MLServer focuses on the **V2 Inference Protocol**, a standardized API also used by other servers.
- This makes it easier to:
  - Plug MLServer into existing API gateways.
  - Swap runtimes or move models between different serving backends without changing clients.

### 5. Integrations with higher-level platforms

- MLServer is used as the underlying model runtime in some **KServe** deployments.
- That means you can:
  - Use MLServer configuration locally in notebooks.
  - Later run the same model definitions under KServe in a production Kubernetes cluster.

These features make MLServer a good fit as a **general-purpose model runtime** in a larger MLOps platform.

In [ ]:
# Advanced feature example: multi-model configuration

# In a real deployment you typically have multiple models, each with
# its own model-settings.json. For illustration, here is what two
# such files might look like.

iris_sklearn_settings = {
    "name": "iris-sklearn",
    "implementation": "mlserver_sklearn.SKLearnModel",
    "parameters": {
        "uri": "model.joblib"
    }
}

xgb_settings = {
    "name": "fraud-xgboost",
    "implementation": "mlserver_xgboost.XGBoostModel",
    "parameters": {
        "uri": "model.bst"
    }
}

print("Example iris-sklearn model-settings.json:\n")
print(json.dumps(iris_sklearn_settings, indent=2))

print("\nExample fraud-xgboost model-settings.json:\n")
print(json.dumps(xgb_settings, indent=2))

# MLServer will discover both models if they live under the same models directory, e.g.:
# models/
#   iris-sklearn/
#     model.joblib
#     model-settings.json
#   fraud-xgboost/
#     model.bst
#     model-settings.json
#
# When you run `mlserver start models`, both models become available via the V2 API.

## Use Cases

### 1. Multi-framework model serving in a single platform

- **Scenario**: Your organization uses scikit‑learn for classic ML, XGBoost for tabular models, and ONNX for exported deep learning models.
- **Pattern**:
  - Package each model in its own folder with a runtime-specific `model-settings.json`.
  - Run one or more MLServer instances that mount a shared `models/` directory.
- **Benefits**:
  - Single operational model for many frameworks.
  - Easier standardization on protocols, logging, and metrics.

### 2. Underlying runtime for KServe deployments

- **Scenario**: You deploy models on Kubernetes using **KServe**.
- **Pattern**:
  - KServe uses MLServer as the **runtime container** for certain model types.
  - Locally, you experiment with MLServer configs in notebooks.
  - In production, you reuse the same configs in KServe `InferenceService` resources.
- **Benefits**:
  - Consistent behavior between local dev and cluster deployments.
  - Clear separation of responsibilities: KServe as control plane, MLServer as data plane.

### 3. Multi-tenant or per-customer models

- **Scenario**: You host per-tenant models (e.g., one model per customer or region).
- **Pattern**:
  - Organize models as `models/customer-a/`, `models/customer-b/`, etc.
  - Each directory has its own artifact and `model-settings.json`.
  - MLServer serves all of them from a single process or a small number of processes.
- **Benefits**:
  - Reduced infrastructure footprint compared to one process per model.
  - Unified monitoring and logging setup.

### 4. Batch scoring via HTTP/gRPC

- **Scenario**: Nightly ETL or streaming jobs need to run inference on large batches.
- **Pattern**:
  - Batch jobs send V2 Inference requests with multiple samples per request.
  - Optionally use MLServer’s batching capabilities (depending on runtime and configuration).
- **Benefits**:
  - Reuse the same serving stack for both online and batch use cases.
  - Centralized management of model versions and rollouts.

### 5. Custom pre/post-processing in Python

- **Scenario**: You need light business logic (feature scaling, enrichment, output formatting) around your model.
- **Pattern**:
  - Implement a custom runtime or hooks in Python.
  - Keep the higher-level application logic in separate services.
- **Benefits**:
  - Flexibility where you need it, without building a full bespoke serving framework yourself.

## Best Practices

### 1. Standardize model packaging

- Define a **clear convention** for how models are stored under `models/`.
- Keep each model in its own directory with:
  - The artifact file (e.g., `model.joblib`, `model.bst`, `model.onnx`).
  - A `model-settings.json` describing runtime, name, and parameters.
- Store model metadata (owner, training run, data snapshot) alongside or in a registry.

### 2. Separate model logic from infrastructure

- Training code should focus on **producing artifacts**; MLServer focuses on **serving**.
- Avoid a lot of business logic in the serving layer—keep that in upstream services or a separate API gateway.
- Use MLServer’s Python extensibility for light pre/post-processing, not full application logic.

### 3. Use explicit versioning and environments

- Version models at the directory level (e.g., `iris-sklearn-v1/`, `iris-sklearn-v2/`).
- Use separate environments (namespaces, clusters, or MLServer instances) for **dev**, **staging**, and **prod**.
- Automate promotion of models between environments via CI/CD.

### 4. Align on the inference protocol

- Treat the V2 Inference API as a **contract** between clients and MLServer.
- Document input/output schemas for each model.
- Add contract tests that:
  - Spin up MLServer locally.
  - Send real V2 requests.
  - Validate response shape and types.

### 5. Make observability part of the design

- Instrument clients and gateways with metrics and tracing.
- Use structured logs with model name, version, and correlation IDs.
- Monitor:
  - Latency (p50/p95/p99)
  - Error rates
  - QPS
  - Resource utilization

### 6. Plan for scale-out, not just scale-up

- Run multiple MLServer replicas behind a load balancer.
- Prefer **stateless** model servers; keep state (artifacts, metadata) in external systems.
- Use autoscaling policies tuned to your traffic patterns.

## Common Pitfalls

### 1. Mismatched model names and URLs

**Symptoms**
- Requests to `/v2/models/<name>/infer` return `NOT_FOUND` or similar errors.

**Causes**
- The `name` in `model-settings.json` does not match the name you use in the URL.
- Multiple models share confusingly similar names.

**How to avoid**
- Treat the `name` field in `model-settings.json` as the **canonical identifier**.
- Document the exact endpoint paths clients should use.

---

### 2. Incorrect runtime implementation

**Symptoms**
- MLServer fails to start a model.
- Logs mention missing classes or import errors for runtimes.

**Causes**
- `implementation` in `model-settings.json` points to the wrong class.
- Required runtime package (e.g., `mlserver-sklearn`) isn’t installed.

**How to avoid**
- Double-check the implementation class name from the MLServer docs.
- Ensure matching runtime packages are installed in the image or environment.

---

### 3. Confusing input/output schemas

**Symptoms**
- Requests fail with validation errors or weird shape/type mismatches.

**Causes**
- Clients send payloads that don’t match what the runtime expects.
- No shared contract for input feature order, types, or shapes.

**How to avoid**
- Define and document a clear **inference schema** for each model.
- Write small smoke tests that send realistic V2 requests and validate responses.

---

### 4. Overloaded single instance

**Symptoms**
- Latency spikes when traffic increases.
- CPU utilization pegs at 100%, or container gets OOM-killed.

**Causes**
- Only one MLServer replica handles all traffic.
- Heavy models share the same instance without enough resources.

**How to avoid**
- Scale out: run multiple MLServer replicas behind a load balancer.
- Use resource requests/limits and separate pools for heavy vs. light models.

---

### 5. Drift between training and serving environments

**Symptoms**
- Model behaves differently in production than in offline experiments.

**Causes**
- Different library versions between training environment and MLServer image.
- Preprocessing logic implemented differently in training vs. serving.

**How to avoid**
- Pin runtime versions in your Dockerfiles and `requirements.txt`.
- Where possible, package preprocessing/post-processing as part of the model or runtime itself.
- Run **end-to-end tests** that use the same MLServer image you deploy to production.

## Performance Optimization

Optimizing MLServer is about matching **hardware, concurrency, and model complexity** to your workload.

### 1. Benchmark first

- Use a simple load generator (Python script, `hey`, `wrk`, Locust) to measure:
  - p50 / p95 / p99 latency
  - Throughput (QPS)
  - CPU and memory usage
- Benchmark each **model type** separately if they differ significantly.

### 2. Tune concurrency and replicas

- Scale **horizontally** by running multiple MLServer replicas.
- Start with 1–2 workers per CPU core for lightweight models; fewer for heavy models.
- Use Kubernetes or your orchestrator’s autoscaler to respond to QPS changes.

### 3. Use batching where appropriate

- Some runtimes support efficient **batch inference** (e.g., ONNX, deep learning models).
- Recommend:
  - Identify models that benefit from batching (matrix-heavy, GPU-backed).
  - Configure clients to send small batches where latency budgets allow.

### 4. Right-size hardware

- For small tree-based / linear models, CPU-only deployments are often sufficient.
- For deep learning models (served via ONNX or PyTorch), GPUs may be necessary.
- Monitor utilization and scale up/down instance sizes accordingly.

### 5. Optimize the model itself

- Use model compression where possible (quantization, pruning).
- Remove unused features and outputs to reduce compute.
- Export models to formats that have efficient runtimes (e.g., ONNX for cross-framework portability).

### 6. Observe and iterate

- Track performance metrics per model and per version.
- Re-benchmark after major changes:
  - New model versions
  - New runtimes or framework versions
  - Changes in traffic patterns

In [ ]:
# Simple latency benchmark for an MLServer endpoint (V2 Inference API)

import time
import statistics
import requests
import numpy as np

MODEL_NAME = "iris-sklearn"  # must match model-settings.json
URL = f"http://localhost:8080/v2/models/{MODEL_NAME}/infer"

# Build a small batch of random inputs (same shape as Iris: 4 features)
inputs = np.random.rand(16, 4).astype("float32")

payload = {
    "inputs": [
        {
            "name": "input-0",
            "shape": list(inputs.shape),
            "datatype": "FP32",
            "data": inputs.flatten().tolist(),
        }
    ]
}


def benchmark(url: str, payload: dict, num_requests: int = 20):
    latencies = []
    for i in range(num_requests):
        start = time.time()
        resp = requests.post(url, json=payload)
        elapsed = time.time() - start
        if not resp.ok:
            print(f"Request {i} failed: {resp.status_code} {resp.text}")
        latencies.append(elapsed)

    print(f"Requests: {num_requests}")
    print(f"Mean latency: {statistics.mean(latencies):.4f} s")
    print(f"p50: {statistics.median(latencies):.4f} s")
    print(f"min: {min(latencies):.4f} s, max: {max(latencies):.4f} s")


# Uncomment after starting MLServer with `mlserver start models`
# benchmark(URL, payload, num_requests=50)

## Production Deployment

MLServer is designed to run as a **containerized microservice**, often managed by an orchestrator like Kubernetes or KServe.

### 1. Docker deployment (single instance)

For local testing or small setups, you can run MLServer directly via Docker.

```bash
# Build an image that contains your models and configs
# (assuming your models/ directory lives alongside this Dockerfile)

FROM python:3.10-slim

# Install MLServer and runtimes
RUN pip install --no-cache-dir mlserver mlserver-sklearn

WORKDIR /app
COPY models/ ./models

# Expose MLServer's default HTTP port
EXPOSE 8080

# Start MLServer pointing at ./models
CMD ["mlserver", "start", "models"]
```

Build and run:

```bash
docker build -t mlserver-iris:latest .
docker run -p 8080:8080 mlserver-iris:latest
```

### 2. Kubernetes deployment (standalone MLServer)

A simple Kubernetes `Deployment` + `Service` might look like:

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mlserver-iris
spec:
  replicas: 2
  selector:
    matchLabels:
      app: mlserver-iris
  template:
    metadata:
      labels:
        app: mlserver-iris
    spec:
      containers:
      - name: mlserver
        image: your-registry/mlserver-iris:latest
        ports:
        - containerPort: 8080
        resources:
          requests:
            cpu: "500m"
            memory: "1Gi"
          limits:
            cpu: "2"
            memory: "2Gi"
---
apiVersion: v1
kind: Service
metadata:
  name: mlserver-iris
spec:
  selector:
    app: mlserver-iris
  ports:
  - name: http
    port: 80
    targetPort: 8080
  type: LoadBalancer
```

### 3. Using MLServer via KServe

In many setups, MLServer runs **inside** KServe’s InferenceService pods:

- You define an `InferenceService` CRD that references a model artifact (e.g., in S3 or a PVC).
- KServe injects MLServer as the data plane container.
- You interact only with the KServe endpoint; MLServer is an implementation detail.

This notebook focuses on direct MLServer usage, but understanding it makes debugging and customization in KServe much easier.

### 4. CI/CD and rollout strategy

- Use CI to build and push updated MLServer images (or update the `models/` volume).
- Use rolling updates in Kubernetes to gradually roll out new versions.
- Tie deployments to model registry events so that **validated models** automatically promote to staging/prod.

## Monitoring and Observability

MLServer should be monitored like any other production microservice: you need **metrics, logs, and (optionally) traces**.

### 1. Key metrics to track

At minimum, capture:

- **Latency**
  - p50 / p95 / p99 for inference requests
  - Per-model, per-version breakdown
- **Error rates**
  - HTTP status codes (4xx, 5xx)
  - Application-level errors (schema mismatches, runtime failures)
- **Throughput (QPS)**
  - Requests per second by model
- **Resource utilization**
  - CPU / memory
  - GPU utilization and memory (if serving DL models)

These can come from:

- Sidecars/agents (Prometheus node exporter, OpenTelemetry collectors)
- Your API gateway or service mesh
- Custom metrics emitted by clients that call MLServer

### 2. Logging best practices

- Use **structured logs** (e.g., JSON) including:
  - Timestamp, request ID / correlation ID
  - Model name and version
  - Status (success / error) and latency
- Centralize logs (e.g., Elasticsearch, Cloud Logging, Loki) and build dashboards.
- Scrub or avoid logging sensitive input data.

### 3. Tracing

If you use distributed tracing:

- Treat MLServer as a **downstream dependency** in your traces.
- Propagate trace IDs from the caller through to the MLServer request.
- Correlate slow spans with model name/version to pinpoint problematic models.

### 4. Model behavior monitoring

Beyond infra metrics, track **model quality over time**:

- Input feature distributions (to detect data drift).
- Output distributions / scores (to detect calibration changes).
- Per-version performance metrics (AUC, accuracy, business KPIs).

This usually requires logging features/predictions from the calling services or a dedicated model-monitoring component, not MLServer alone.

## Troubleshooting

### Issue 1: Model not found (`NOT_FOUND`-style errors)

**Symptoms**
- Requests to `/v2/models/<name>/infer` fail.
- Logs indicate that the model is not loaded.

**Likely causes**
- The `name` in `model-settings.json` does not match `<name>` in the URL.
- Model directory is missing or mounted at a different path than expected.

**How to fix**
- Verify that `model-settings.json` exists under `models/<model-name>/`.
- Ensure the `name` field matches the URL path.
- Check MLServer logs at startup to confirm which models were discovered.

---

### Issue 2: Runtime import or initialization errors

**Symptoms**
- MLServer fails to start, or specific models fail to load.
- Logs mention missing classes or modules.

**Likely causes**
- `implementation` in `model-settings.json` points to a non-existent class.
- Required runtime package (e.g., `mlserver-sklearn`) is not installed.

**How to fix**
- Double-check runtime docs for the correct implementation class.
- Confirm runtime packages are installed in your image or environment.
- Run a small Python snippet to import the runtime class directly.

---

### Issue 3: Shape or datatype mismatches

**Symptoms**
- Inference requests fail with validation errors.
- Outputs are misaligned with what the client expects.

**Likely causes**
- V2 Inference payloads (shape, `datatype`, etc.) do not match the model.
- Training and serving use different feature encodings.

**How to fix**
- Inspect the model’s expected input shape and dtype.
- Standardize on a single schema and document it.
- Write smoke tests that send realistic inputs through MLServer.

---

### Issue 4: High latency under load

**Symptoms**
- Latency spikes when traffic increases.
- Container CPU or memory maxes out.

**Likely causes**
- Too few MLServer replicas for incoming QPS.
- Heavy models sharing a single instance.

**How to fix**
- Scale out: add more replicas behind a load balancer.
- Split heavy models into separate deployments.
- Tune resource requests/limits and autoscaling policies.

---

### Issue 5: Drift between training and serving behavior

**Symptoms**
- Model behaves differently in production than in offline experiments.

**Likely causes**
- Different library versions between training env and MLServer image.
- Inconsistent preprocessing/post-processing logic.

**How to fix**
- Pin dependencies in training and serving environments.
- Where possible, embed preprocessing into the model or runtime.
- Run end-to-end tests that use the same image you deploy to production.

## Comparison with Alternatives

MLServer is one option among several model-serving systems.

| Capability / Tool            | MLServer                               | TensorFlow Serving                 | Triton Inference Server                     | TorchServe                          | BentoML / OpenLLM                        |
|-----------------------------|----------------------------------------|------------------------------------|---------------------------------------------|-------------------------------------|------------------------------------------|
| Primary focus               | General ML runtime, V2 protocol        | TensorFlow `SavedModel`s           | Multi-framework, GPU-optimized              | PyTorch models                      | Application-centric ML/LLM serving       |
| Implementation language     | Python                                 | C++                                | C++                                         | Java + Python                       | Python                                   |
| Multi-framework             | Yes (via runtimes)                     | Mostly TF                          | Yes (TF, Torch, ONNX, etc.)                 | Limited (PyTorch-centric)           | Yes                                      |
| V2 Inference Protocol       | Yes                                    | No (different APIs)                | Yes                                         | No                                   | Optional / via wrappers                  |
| Extensibility               | Python runtimes + hooks                | C++ extensions                     | C++ backends                                | Handlers in Python                  | Full Python services                     |
| Integration with KServe     | Yes (used as runtime)                  | Yes (as a backend)                 | Yes                                         | Yes                                  | Yes (via custom predictors)             |
| Best fit                    | General-purpose runtime in platforms   | TF-centric production systems      | GPU-heavy, multi-framework DL/LLM           | PyTorch-only orgs                  | App logic + model serving combined       |

### When to choose MLServer

Choose MLServer when:

- You want a **general-purpose model runtime** that supports many frameworks.
- You care about **protocol compatibility** (V2 Inference API) and platform integration.
- You’re building or using a control plane (e.g., KServe) and need a flexible data-plane runtime.

Consider alternatives when:

- You serve only TensorFlow models and want a battle-tested TF-centric stack (TensorFlow Serving).
- You need extreme GPU optimizations for deep learning/LLM workloads (Triton, vLLM, TGI).
- You want to build rich, Python-centric applications with tight model + business logic integration (BentoML, FastAPI, Mosec).

## Resources

### Official documentation

- **MLServer documentation**  
  https://mlserver.readthedocs.io/
- **MLServer GitHub repository**  
  https://github.com/SeldonIO/MLServer

### Tutorials and guides

- Getting started with MLServer (search terms):
  - "MLServer getting started"
  - "MLServer sklearn tutorial"
  - "MLServer KServe integration"
- Example topics:
  - Multi-model deployments with MLServer
  - Serving ONNX models via MLServer
  - Custom runtimes and pre/post-processing hooks

### Community resources

- Seldon community and GitHub issues:  
  https://github.com/SeldonIO/MLServer/issues
- Stack Overflow questions tagged `seldon` or `kserve`.

### Related technologies

- **KServe** – Kubernetes-native model serving platform that often uses MLServer as a runtime.  
- **TensorFlow Serving** – TF-specific serving stack for SavedModels.  
- **Triton Inference Server** – multi-framework, GPU-optimized serving engine.  
- **TorchServe** – serving for PyTorch models.  
- **BentoML / OpenLLM** – higher-level serving framework and LLM-oriented serving.  
- **vLLM, TGI** – high-performance LLM-specific serving stacks.